# Lakeside heating DSM — sklearn hybrid (human SoT)

**Real deal only.** No BAS proxy, no synthetic zone temps, no quarantined hourly ship.

| Component | What | Provenance |
|---|---|---|
| A | Real BAS 15-min baseline (7 outs) | `REAL_BAS_15MIN` |
| B | Paired E+ IdealLoads+COP deltas | `ENERGYPLUS_NATIVE_RUN` → delta |
| C | Hybrid 96-step walk | `HYBRID_SCREENING` |

Ship surface: `desktop/artifacts/hybrid_dsm_96_v1_walk.json` (not `heating_dsm_hourly_v1.onnx`).

Set `LAKESIDE_SITE_ROOT` before running.

## 0 · Setup

In [ ]:
from pathlib import Path
import sys, json, os, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = Path("..").resolve()
if not (ROOT / "ml").is_dir():
    ROOT = Path(".").resolve()
sys.path.insert(0, str(ROOT / "ml"))

from artifact_paths import artifact_paths, train_parquet_path
from notebook_proof import prove_native_farm_load, prove_real_store_load
from notebook_plots import save_fig

PATHS = artifact_paths()
PATHS["figures"].mkdir(parents=True, exist_ok=True)
SITE = Path(os.environ.get("LAKESIDE_SITE_ROOT", r"C:\Users\ben\OneDrive\Desktop\testing\sp_creekside"))
print("ROOT", ROOT)
print("SITE", SITE)
print("paired farm", train_parquet_path())

## 1 · Proof — real BAS 15-min store (component A)

In [ ]:
real_df, real_meta = prove_real_store_load(site=SITE)
assert (real_df["provenance"] == "REAL_BAS_15MIN").all()
print("real rows", len(real_df), "days", real_df["day"].nunique())

## 2 · Proof — paired native E+ farm (component B data)

In [ ]:
farm_df, farm_meta = prove_native_farm_load(root=ROOT, paths=PATHS, site=SITE)
assert train_parquet_path().resolve() == PATHS["eplus_paired"].resolve()
assert "arm" in farm_df.columns and set(farm_df["arm"].unique()) >= {"baseline", "dsm"}
zcols = [c for c in farm_df.columns if c.startswith("zone_temp_") and c.endswith("_f")]
assert len(zcols) == 6, zcols
print("farm OK", farm_meta["farm_path"])

## 3 · EDA charts — measured kW vs OAT + paired arms

In [ ]:
winter = real_df[real_df["month"].isin([11, 12, 1, 2, 3])].copy()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(winter["oat_f"], winter["facility_kw"], s=4, alpha=0.25, c="#4C78A8")
axes[0].set_xlabel("OAT °F"); axes[0].set_ylabel("facility kW")
axes[0].set_title("Real BAS 15-min — winter kW vs OAT")
axes[0].spines["top"].set_visible(False); axes[0].spines["right"].set_visible(False)

for arm, color in (("baseline", "#4C78A8"), ("dsm", "#F58518")):
    sub = farm_df[farm_df["arm"] == arm]
    axes[1].scatter(sub["oat_f"], sub["facility_kw"], s=8, alpha=0.35, label=arm, c=color)
axes[1].set_xlabel("OAT °F"); axes[1].set_ylabel("IdealLoads+COP kW")
axes[1].set_title("Paired E+ farm — baseline vs DSM")
axes[1].legend(frameon=False)
axes[1].spines["top"].set_visible(False); axes[1].spines["right"].set_visible(False)
plt.tight_layout()
save_fig(PATHS["figures"] / "hybrid_eda_kw_oat.png", fig)
plt.show()

## 4 · Model cards — ExtraTrees baseline + RF delta (no re-ship of hourly v1)

In [ ]:
base_card = json.loads(PATHS["real_baseline_card"].read_text(encoding="utf-8"))
delta_card = json.loads(PATHS["delta_card"].read_text(encoding="utf-8"))
assert base_card.get("honesty") == "HYBRID_SCREENING"
assert delta_card.get("honesty") == "HYBRID_SCREENING"
assert base_card.get("provenance") == "REAL_BAS_15MIN"

cv = base_card["cv_teacher_forced"]
rows = []
for fam, m in cv.items():
    rows.append({
        "family": fam,
        "mae": m.get("facility_kw_mae", m.get("facility_kw", {}).get("mae")),
        "mae_peak_05_09": m.get("facility_kw_mae_peak_05_09", m.get("facility_kw", {}).get("mae_peak_05_09")),
        "zone_temp_mae_mean": m.get("zone_temp_mae_mean"),
    })
lb = pd.DataFrame(rows).sort_values("mae_peak_05_09").reset_index(drop=True)
display(Markdown(f"### Real baseline champion: **{base_card['champion']}**"))
display(lb.round(3))

dcv = delta_card["cv_teacher_forced"]
dlb = pd.DataFrame([{"family": k, **v} for k, v in dcv.items()]).sort_values("mae_delta_kw_peak")
display(Markdown(f"### E+ delta champion: **{delta_card['champion']}** (smoke/paired farm)"))
display(dlb.round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].barh(lb["family"], lb["mae_peak_05_09"], color="#2a9d8f")
axes[0].set_xlabel("Peak MAE kW"); axes[0].set_title("Real baseline bake-off")
axes[1].barh(dlb["family"], dlb["mae_delta_kw_peak"], color="#e76f51")
axes[1].set_xlabel("Peak ΔkW MAE"); axes[1].set_title("E+ delta bake-off")
for ax in axes:
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
plt.tight_layout()
save_fig(PATHS["figures"] / "hybrid_sklearn_leaderboards.png", fig)
plt.show()

## 5 · Hybrid 96-step walk (desktop ship evidence)

In [ ]:
walk_path = PATHS["hybrid_walk"]
assert walk_path.is_file(), f"missing {walk_path} — run scripts/promote_hybrid_ship.py"
walk = json.loads(walk_path.read_text(encoding="utf-8"))
assert walk.get("honesty") == "HYBRID_SCREENING"
assert len(walk["steps"]) == 96
steps = pd.DataFrame(walk["steps"])
steps["hour"] = steps["step_15"] / 4.0

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(steps["hour"], steps["baseline_facility_kw"], label="real baseline", lw=1.8)
ax.plot(steps["hour"], steps["hybrid_facility_kw"], label="hybrid DSM", lw=1.8)
ax.set_xlabel("Local hour"); ax.set_ylabel("kW")
ax.set_title("Hybrid 96-step — baseline vs DSM (HYBRID_SCREENING)")
ax.legend(frameon=False)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
plt.tight_layout()
save_fig(PATHS["figures"] / "hybrid_96_walk_kw.png", fig)
plt.show()

display(Markdown("### Walk summary"))
display(pd.Series(walk["summary"]).to_frame("value"))
print("desktop walk", walk_path)
print("champions", walk.get("champion_baseline"), "+", walk.get("champion_delta"))

## Honesty

- IdealLoads + fixed COP ≠ GSHP plant.
- Real BAS and E+ rows are **never** concatenated for training.
- `HYBRID_SCREENING` until field DSM trials.
- Quarantined hourly `heating_dsm_hourly_v1.*` is **not** the ship path.